In [0]:
spark.conf.set(
  "fs.azure.account.key.datalakeprateek123.dfs.core.windows.net",
  "Im6Exl5aA6SEUPOaSr0rWIY1fV9Ap2KnHQNV1GriUnMGtZxzpkOuw9dUZJ0fA2cBNSA45lx4pDZe+AStxpAktQ=="
)

In [0]:
data = [
    (1,101,"India",100,"2024-01-01"),
    (2,102,"US",200,"2024-01-01"),
    (3,103,"India",150,"2024-01-02"),
    (4,104,"UK",300,"2024-01-02"),
    (5,101,"India",120,"2024-01-03")
]

columns = ["order_id","customer_id","region","sales","date"]

df = spark.createDataFrame(data, columns)

df.display()

order_id,customer_id,region,sales,date
1,101,India,100,2024-01-01
2,102,US,200,2024-01-01
3,103,India,150,2024-01-02
4,104,UK,300,2024-01-02
5,101,India,120,2024-01-03


In [0]:
df.write.mode("overwrite").csv(
  "abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/"
)

In [0]:
display(dbutils.fs.ls("abfss://bronze@datalakeprateek123.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/,sales/,0,1776060009000


In [0]:
df_bronze = spark.read.csv(
  "abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/",
  header=True,
  inferSchema=True
)

df_bronze.display()

4,104,UK,300,2024-01-02
5,101,India,120,2024-01-03


In [0]:
df_silver = df_bronze.dropDuplicates().na.fill("NA")

In [0]:
df_silver.write.mode("overwrite").format("delta").save(
  "abfss://silver@datalakeprateek123.dfs.core.windows.net/sales/"
)

In [0]:
df_bronze.printSchema()

root
 |-- 4: integer (nullable = true)
 |-- 104: integer (nullable = true)
 |-- UK: string (nullable = true)
 |-- 300: integer (nullable = true)
 |-- 2024-01-02: date (nullable = true)



In [0]:
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/")

In [0]:
df_bronze.display()

4,104,UK,300,2024-01-02
5,101,India,120,2024-01-03


In [0]:
df_bronze.printSchema()


root
 |-- 4: integer (nullable = true)
 |-- 104: integer (nullable = true)
 |-- UK: string (nullable = true)
 |-- 300: integer (nullable = true)
 |-- 2024-01-02: date (nullable = true)



In [0]:
dbutils.fs.rm("abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/", True)

True

In [0]:
df.write.mode("overwrite") \
  .option("header", "true") \
  .csv("abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/")

In [0]:
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@datalakeprateek123.dfs.core.windows.net/sales/")

In [0]:
df_bronze.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- date: date (nullable = true)



In [0]:
df_silver = df_bronze.dropDuplicates().na.fill("NA")

In [0]:
from pyspark.sql.functions import sum

df_gold = df_silver.groupBy("region").agg(sum("sales").alias("total_sales"))